# Train classification model

Imports

In [5]:
import torch
from torch import nn
import torchvision.transforms as transforms
import sys
import os
import torch.nn.functional as F
from ray.train import Checkpoint, get_checkpoint
from ray import tune
from tqdm.auto import tqdm
# Absolute path to your project's src folder
src_path = os.path.abspath(os.path.join("..", "src"))

# Add to sys.path if not already present
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(sys.path)

from data import get_data_loaders, show_image
from models.classification_model import ClassificationModel
from parameter_optimization import optimize_parameters, get_best_config
import config

['/Users/bevanslabbert/Documents/GitHub/pid-radast/src', '/Users/bevanslabbert/Documents/GitHub/pid-radast/.venv/lib/python3.13/site-packages/ray/thirdparty_files', '/opt/homebrew/Cellar/python@3.13/3.13.5/Frameworks/Python.framework/Versions/3.13/lib/python313.zip', '/opt/homebrew/Cellar/python@3.13/3.13.5/Frameworks/Python.framework/Versions/3.13/lib/python3.13', '/opt/homebrew/Cellar/python@3.13/3.13.5/Frameworks/Python.framework/Versions/3.13/lib/python3.13/lib-dynload', '', '/Users/bevanslabbert/Documents/GitHub/pid-radast/.venv/lib/python3.13/site-packages', '/var/folders/xl/d7wznkt15fq8wspgq66fyc9m0000gn/T/tmpo_75l0er']


Data Initialization

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
])

trainloader, testloader = get_data_loaders(
    "Mirabest",
    transforms.Compose([
        transforms.ToTensor(), # to range [0,1]
        transforms.Normalize([0.5], [0.5]) # 0 centers
        ]),
        config.CLASSIFICATION_MODEL["batch_size"])
    # transform_train) TODO: this is the correct transformer for data augmentation, not working atm

print(next(iter(trainloader))[0].shape)
print(next(iter(trainloader)))

Files already downloaded and verified
Files already downloaded and verified
torch.Size([2, 1, 32, 32])
[tensor([[[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]],


        [[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]]]), tensor([0, 1])]


In [9]:
import torch.nn.functional as F

def pgd_attack(model, images, labels, eps=8/255, alpha=2/255, iters=10):
    device = next(model.parameters()).device

    images = images.clone().detach().to(device).float()
    labels = labels.clone().detach().to(device)
    ori_images = images.clone().detach()

    for _ in range(iters):
        images.requires_grad = True
        outputs = model(images)
        loss = F.cross_entropy(outputs, labels)
        model.zero_grad()
        loss.backward()
        grad = images.grad.sign()

        images = images + alpha * grad
        delta = torch.clamp(images - ori_images, min=-eps, max=eps)
        images = torch.clamp(ori_images + delta, min=0, max=1).detach()

    return images


Parameter optimization

In [ ]:
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)

tuned_config = get_best_config(ClassificationModel, parent_dir)
print(tuned_config)
model = ClassificationModel(tuned_config)
criterion = tuned_config['criterion_class']()
optimizer = tuned_config['optimizer_class'](model.parameters(), lr=tuned_config['lr'])
batch_size = tuned_config['batch_size']
# TODO: incorporate adversarially attacked images loss 
for epoch in range(config.CLASSIFICATION_MODEL["num_epochs"]):  # loop over the dataset multiple times

    print(f'Epoch {epoch+1}')
    running_loss = 0.0
    for i, data in tqdm(trainloader, 0):
        # get the inputs
        images, labels = data

        # Generate adversarial examples
        # adv_images = pgd_attack(model, images, labels)

        # Forward on clean and adversarial
        outputs_clean = model(images)
        # outputs_adv = model(adv_images)

        # Combined loss
        loss_clean = F.cross_entropy(outputs_clean, labels)
        # loss_adv = F.cross_entropy(outputs_adv, labels)
        # loss = 0.5 * (loss_clean + loss_adv)
        loss = loss_clean

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch} | Loss: {total_loss/len(trainloader):.4f}")

print('Finished Training')



2025-08-12 17:50:26,478	WARNING experiment_analysis.py:180 -- Failed to fetch metrics for 1 trial(s):
- train__5efc7_00029: FileNotFoundError('Could not fetch metrics for train__5efc7_00029: both result.json and progress.csv were not found at /Users/bevanslabbert/Documents/GitHub/pid-radast/tuning_results/ClassificationModel/train__5efc7_00029_29_batch_size=8,lr=0.0001,optimizer_class=ref_ph_626dc8cf_2025-08-02_15-41-19')


Getting config from /Users/bevanslabbert/Documents/GitHub/pid-radast/tuning_results/ClassificationModel
{'lr': 0.0016409286730647919, 'optimizer_class': <class 'torch.optim.adamw.AdamW'>, 'model_class': <class 'models.classification_model.ClassificationModel'>, 'criterion_class': <class 'torch.nn.modules.loss.CrossEntropyLoss'>, 'dataset': 'MiraBest', 'batch_size': 8}
Epoch 1


  0%|          | 0/347 [00:01<?, ?it/s]

RuntimeError: Expected 3D (unbatched) or 4D (batched) input to conv2d, but got input of size: []

In [ ]:
# get data
dataiter = iter(testloader)
images, labels = next(dataiter)

# get predictions
output = model(images)
_, predicted = torch.max(output, 1)

In [ ]:
correct = 0
total = 0
with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print('Accuracy of the network on the 88 test images: %d %%' % (100 * correct / total))

Accuracy of the network on the 88 test images: 68 %


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adagrad(optimized_model.parameters(), lr=0.01)

for epoch in range(config.CLASSIFICATION_MODEL["num_epochs"]):  # loop over the dataset multiple times
    print(f'Epoch {epoch+1}')
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = optimized_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % print_num == (print_num-1):    # print every 50 mini-batches
            print('[%d, %5d] loss: %.3f' %
                  (epoch + 1, i + 1, running_loss / print_num))
            running_loss = 0.0

print('Finished Training')

NameError: name 'optimized_model' is not defined

In [ ]:
# get data
dataiter = iter(testloader)
images, labels = next(dataiter)

# get predictions
output = optimized_model(images)
_, predicted = torch.max(output, 1)

correct = 0
total = 0
with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = optimized_model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print('Accuracy of the network on the 88 test images: %d %%' % (100 * correct / total))


Accuracy of the network on the 88 test images: 74 %


Train

Test